# Session 1b — Photometric Redshifts in CEERS with Regression

In the first notebook we learned least squares, linear and polynomial regression, train/test splits, overfitting and regularization on a toy data set. Now we apply the same tools to a real problem: **predicting galaxy redshifts from their photometry**.

We use the public **CEERS DR1 photometric catalogue** (Cox et al. 2025, [arXiv:2510.08743](https://arxiv.org/abs/2510.08743)): 7 JWST/NIRCam bands (F115W–F444W) and 7 HST ACS/WFC3 bands (F435W–F160W) in the Extended Groth Strip.

**Our target.** The catalogue contains photometric redshifts from the template-fitting code **LePHARE** (`LP_Z_MED`). We use these as our labels, so we are training a regression model to **emulate a template-fitting code**. That is a genuinely useful thing to do: for the billions of galaxies that Euclid and Roman will observe, a trained model is orders of magnitude faster than fitting templates to every object. But keep in mind throughout that our "truth" is itself an estimate.

**Plan**

1. Download and load the catalogue
2. A first look at the data
3. Cleaning: flags, stars, artifacts, low signal-to-noise, unreliable labels
4. Choosing and building features (magnitudes and colours)
5. Linear regression and photo-z metrics
6. Polynomial features, regularization, and a better target variable
7. Where does the model fail, and how could we do better?

Cells marked **✏️ Exercise** are for you to complete.

In [ ]:
#@title Setup (run me) {display-mode: "form"}
import sys
if "google.colab" in sys.modules:
    !pip install -q astropy gdown

import re
from pathlib import Path
import gdown

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from astropy.table import Table

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# One-hue sequential ramp (light = low redshift, dark = high redshift)
ZCMAP = LinearSegmentedColormap.from_list("zramp", ["#b7d3f6", "#5598e7", "#1c5cab", "#0d366b"])

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5), "figure.dpi": 110,
    "font.size": 12, "axes.titlesize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "legend.frameon": False,
})

## 1. Download and load the catalogue

The catalogue is downloaded once and cached in a local `DATA/` folder. The file is large, so the first download can take a few minutes.

In [ ]:
FILE_ID = "1zBV9flD297bxaaG7BNG3E_VPN51DzLxE"
LOCAL = Path("DATA") / "ceers_cat_v1.0_subset.fits"

if not LOCAL.exists():
    gdown.download(id=FILE_ID, output=str(LOCAL), quiet=False)
else:
    print(f"Using cached catalogue: {LOCAL}  ({LOCAL.stat().st_size/1e6:.1f} MB)")

In [ ]:
cat = Table.read(LOCAL)
print(f"{len(cat)} sources, {len(cat.colnames)} columns")

# The circular-aperture columns (…_FLUX_APER etc.) hold 12 values per source.
# pandas wants one value per cell, so we keep only the 1-D columns.
one_d = [name for name in cat.colnames if cat[name].ndim == 1]
df = cat[one_d].to_pandas()
print(f"kept {len(one_d)} one-dimensional columns")
df.head()

The catalogue has one set of columns per filter, named `F###_FLUX`, `F###_FLUXERR_EMP`, and so on (see the README). All fluxes are in **nJy**. Let's find the filters, order them by wavelength, and define a few helpers we will use throughout.

AB magnitudes follow from fluxes in nJy as

$$m_\mathrm{AB} = 31.4 - 2.5\log_{10}\left(f_\nu / \mathrm{nJy}\right).$$

In [ ]:
flux_cols = [c for c in df.columns if c.endswith("_FLUX")]

# Approximate pivot wavelengths in micron, keyed by the three digits in the filter name
WAVE = {"435": 0.435, "606": 0.606, "814": 0.814,                     # HST/ACS
        "105": 1.05, "125": 1.25, "140": 1.40, "160": 1.60,           # HST/WFC3
        "115": 1.15, "150": 1.50, "200": 2.00, "277": 2.77,           # JWST/NIRCam
        "356": 3.56, "410": 4.10, "444": 4.44}
NIRCAM = {"115", "150", "200", "277", "356", "410", "444"}

def fid(band):
    return re.search(r"\d{3}", band).group()

bands = sorted([c[:-len("_FLUX")] for c in flux_cols], key=lambda b: WAVE[fid(b)])
wave = {b: WAVE[fid(b)] for b in bands}
instrument = {b: "NIRCam" if fid(b) in NIRCAM else "HST" for b in bands}

def band(num):
    # look up the catalogue name of a filter from its number, e.g. band("277") -> "F277W"
    return next(b for b in bands if fid(b) == num)

def flux_to_mag(f):
    f = np.asarray(f, dtype=float)
    return 31.4 - 2.5*np.log10(np.where(f > 0, f, np.nan))

def has_data(d, b):
    # The README: FLUXERR_SE == 0 and/or FLUXERR_EMP == NaN means the source is outside that filter's footprint
    e_emp, e_se = d[f"{b}_FLUXERR_EMP"], d[f"{b}_FLUXERR_SE"]
    return np.isfinite(d[f"{b}_FLUX"]) & np.isfinite(e_emp) & (e_emp > 0) & (e_se > 0)

print("filters, blue to red:")
for b in bands:
    print(f"  {b:6s} {wave[b]:5.2f} µm  {instrument[b]}")

## 2. A first look

Before any modelling, **look at the data**. How many sources are there, where are they on the sky, which filters cover which part of the field, and what do the redshift labels look like?

In [ ]:
z_all = df["LP_Z_MED"]
print(f"LP_Z_MED: {np.isfinite(z_all).sum()} finite values, "
      f"{(~np.isfinite(z_all)).sum()} NaN, {(z_all < 0).sum()} negative (placeholder values)")
print(z_all[np.isfinite(z_all) & (z_all >= 0)].describe().round(2))

fig, ax = plt.subplots()
ax.hist(z_all[(z_all >= 0) & (z_all < 16)], bins=np.arange(0, 15.01, 0.1))
ax.set(xlabel="LePHARE photo-z  (LP_Z_MED)", ylabel="number of sources",
       title="Redshift labels, full catalogue");

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ok = (df.STAR_FLAG == 0) & (df.BAD_REGION_FLAG == 0)
ax.scatter(df.RA[ok], df.DEC[ok], s=0.3, color="gray", alpha=0.4, rasterized=True, label="unflagged")
ax.scatter(df.RA[df.BAD_REGION_FLAG != 0], df.DEC[df.BAD_REGION_FLAG != 0], s=0.6,
           color="C1", rasterized=True, label="BAD_REGION_FLAG")
ax.scatter(df.RA[df.STAR_FLAG != 0], df.DEC[df.STAR_FLAG != 0], s=0.6,
           color="C0", rasterized=True, label="STAR_FLAG (bright-star mask)")
ax.invert_xaxis()                                # RA increases to the left on the sky
ax.set(xlabel="RA [deg]", ylabel="Dec [deg]", title="CEERS sources on the sky")
ax.set_aspect(1/np.cos(np.deg2rad(df.DEC.median())))
ax.legend(markerscale=12, loc="lower left");

The filters do not all cover the same area: the NIRCam pointings define the catalogue, but the HST imaging has its own, different footprints. The next plot shows, filter by filter, where the catalogue actually has a measurement.

In [ ]:
ncol = 7
nrow = int(np.ceil(len(bands) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.9*nrow), sharex=True, sharey=True)
extent = (df.RA.max(), df.RA.min(), df.DEC.min(), df.DEC.max())
for ax, b in zip(axes.flat, bands):
    m = has_data(df, b)
    ax.hexbin(df.RA[m], df.DEC[m], gridsize=45, cmap="Blues", mincnt=1,
              extent=(df.RA.min(), df.RA.max(), df.DEC.min(), df.DEC.max()))
    ax.set_title(f"{b}  ({m.mean():.0%})", fontsize=11)
    ax.grid(False)
for ax in axes.flat[len(bands):]:
    ax.set_visible(False)
axes.flat[0].invert_xaxis()
fig.suptitle("Where does each filter have data?  (fraction of all sources in brackets)", y=1.01)
fig.tight_layout();

**Discussion**

* Which filters cover the full field, and which only part of it?
* If we use a filter with partial coverage as a feature, what happens to the galaxies outside its footprint? What options do we have?
* The redshift distribution has a long tail to high $z$. How many galaxies are there above $z = 6$? What do you expect that means for a regression model?

## 3. Cleaning the sample

A catalogue is not a clean training set. Before we fit anything we remove:

1. **flagged regions** — sources near bright stars, and sources where not all filters have full coverage (recommended in the README),
2. **stars** — they have no redshift, and a model that "predicts" one for them is meaningless,
3. **artifacts** — spurious detections, fragments, diffraction spikes,
4. **low signal-to-noise sources** — noisy colours *and* unreliable labels,
5. **unreliable labels** — sources where LePHARE itself is very uncertain about the redshift.

We keep track of how many sources survive each step.

### ✏️ Exercise 1 — Flags

The README recommends keeping only sources with `STAR_FLAG == 0` and `BAD_REGION_FLAG == 0`. Create the boolean mask `good_flags`.

Note that `STAR_FLAG` marks sources inside the *mask around bright stars* — it does **not** identify stars in general. We deal with those next.

In [ ]:
funnel = [("full catalogue", len(df))]

# YOUR CODE HERE


funnel.append(("flags", good_flags.sum()))
print(f"{good_flags.sum()} of {len(df)} sources survive the flag cuts ({good_flags.mean():.0%})")

### Stars

Source Extractor's `CLASS_STAR` is a neural-network estimate of how point-like a source is (1 = star, 0 = galaxy). Point sources also have a distinctive relation between size and brightness: at fixed magnitude they are as compact as the PSF allows. Both are only reliable for reasonably bright sources — at the faint end everything looks like a blob and `CLASS_STAR` drifts to intermediate values.

In [ ]:
REF = band("200")                                # a short-wavelength NIRCam band
cs_col = f"{REF}_CLASS_STAR" if f"{REF}_CLASS_STAR" in df.columns else \
         [c for c in df.columns if c.endswith("CLASS_STAR")][0]
print("using", cs_col)

d = df[good_flags]
mag_ref = flux_to_mag(d[f"{REF}_FLUX"])
cs = d[cs_col]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
hb = ax1.hexbin(mag_ref, cs, gridsize=70, bins="log", mincnt=1, cmap="Blues",
                extent=(16, 31, 0, 1))
ax1.set(xlabel=f"{REF} magnitude", ylabel=cs_col, title="Stellarity vs. brightness")

pointlike = cs > 0.8
ax2.scatter(mag_ref[~pointlike], np.log10(d.ISO_AREA[~pointlike]), s=0.3, color="gray",
            alpha=0.3, rasterized=True, label=f"{cs_col} ≤ 0.8")
ax2.scatter(mag_ref[pointlike], np.log10(d.ISO_AREA[pointlike]), s=3, color="C1",
            rasterized=True, label=f"{cs_col} > 0.8")
ax2.set(xlim=(16, 31), xlabel=f"{REF} magnitude", ylabel="log$_{10}$ ISO_AREA [pix]",
        title="Size vs. brightness")
ax2.legend(markerscale=6)
fig.colorbar(hb, ax=ax1, label="number of sources");

Stars also have distinctive **colours**. Cool stars and brown dwarfs, in particular, have spectra shaped by molecular absorption that no galaxy template reproduces. In a colour–colour diagram, the point-like sources separate from the galaxy cloud:

In [ ]:
b1, b2, b3, b4 = band("115"), band("200"), band("277"), band("444")
col_a = flux_to_mag(d[f"{b1}_FLUX"]) - flux_to_mag(d[f"{b2}_FLUX"])
col_b = flux_to_mag(d[f"{b3}_FLUX"]) - flux_to_mag(d[f"{b4}_FLUX"])
bright = mag_ref < 25

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(col_b[bright & ~pointlike], col_a[bright & ~pointlike], s=1, color="gray",
           alpha=0.3, rasterized=True, label="galaxy-like")
ax.scatter(col_b[bright & pointlike], col_a[bright & pointlike], s=8, color="C1",
           label=f"point-like ({cs_col} > 0.8)")
ax.set(xlim=(-1.5, 2.5), ylim=(-1.5, 2.5), xlabel=f"{b3} − {b4}", ylabel=f"{b1} − {b2}",
       title=f"Colour–colour diagram, {REF} < 25")
ax.legend(markerscale=3);

### ✏️ Exercise 2 — Remove stars and artifacts

1. Define `is_star`: bright (`mag_ref` below some limit) **and** point-like (`CLASS_STAR` above some threshold). Use the plots above to choose the numbers, and be ready to justify them.
2. Define `is_artifact`: very small detections (`ISO_AREA` of only a few pixels — the hot-mode detection used a minimum area of 5 pixels) and extremely elongated sources (`ELLIPTICITY` close to 1), which are often diffraction spikes or image edges.

Why is it safer to only classify *bright* sources as stars?

In [ ]:
# YOUR CODE HERE


keep = ~is_star & ~is_artifact
print(f"stars removed    : {is_star.sum()}")
print(f"artifacts removed: {(is_artifact & ~is_star).sum()}")
d = d[keep]
funnel.append(("stars & artifacts", len(d)))

### Signal-to-noise and label quality

Two more cuts, both with a trade-off:

* **Signal-to-noise.** For faint sources the colours are noisy — and so is the LePHARE redshift we want to learn. We require a minimum S/N in F277W, one of the bands that make up the detection image. We use the **empirical** errors (`_FLUXERR_EMP`), which the README recommends since they include correlated noise.
* **Label quality.** `LP_Z_MED_LOW` and `LP_Z_MED_HIGH` are the 16th and 84th percentiles of the LePHARE redshift PDF. If that interval is very wide, the label is not worth learning from. We use the half-width relative to $(1+z)$.

Every cut makes the training set cleaner — but also **less representative** of the galaxies we will eventually want to apply the model to.

In [ ]:
REF_SN = band("277")
sn = d[f"{REF_SN}_FLUX"] / d[f"{REF_SN}_FLUXERR_EMP"]
z = d["LP_Z_MED"]
zwidth = 0.5*(d["LP_Z_MED68_HIGH"] - d["LP_Z_MED68_LOW"]) / (1 + z)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.hist(np.log10(sn[sn > 0]), bins=100)
ax1.set(xlabel=f"log$_{{10}}$ S/N in {REF_SN}", ylabel="number of sources", title="Signal-to-noise")
ax2.hexbin(np.log10(sn.clip(lower=0.1)), zwidth.clip(upper=1), gridsize=70, bins="log",
           mincnt=1, cmap="Blues", extent=(-1, 3.5, 0, 1))
ax2.set(xlabel=f"log$_{{10}}$ S/N in {REF_SN}", ylabel=r"PDF half-width / (1+z)",
        title="Fainter sources have less certain labels");

In [ ]:
SN_MIN = 10          # minimum S/N in the reference band
ZWIDTH_MAX = 0.1     # maximum half-width of the 68% redshift interval, relative to (1+z)
Z_MAX = 15

good = (sn >= SN_MIN) & np.isfinite(z) & (z >= 0) & (z <= Z_MAX) & (zwidth < ZWIDTH_MAX)
d = d[good]
funnel.append((f"S/N ≥ {SN_MIN}, good labels", len(d)))
print(f"{len(d)} sources left")

**Try it:** re-run the two cells above with `SN_MIN = 5` and `ZWIDTH_MAX = 0.3`. How does the redshift distribution of the surviving sample change? Which galaxies do the cuts preferentially remove?

### Which filters can we use?

A regression model needs **the same features for every galaxy**. Let's check the coverage of each filter within our clean sample, and keep only the filters that are available for almost everyone.

In [ ]:
coverage = pd.Series({b: has_data(d, b).mean() for b in bands})
COVERAGE_MIN = 0.9
use_bands = [b for b in bands if coverage[b] >= COVERAGE_MIN]

fig, ax = plt.subplots(figsize=(9, 4))
cols = ["C0" if instrument[b] == "NIRCam" else "C1" for b in bands]
ax.bar(range(len(bands)), coverage.values, color=cols, width=0.75)
ax.axhline(COVERAGE_MIN, color="black", ls="--", lw=1)
ax.set_xticks(range(len(bands)), bands, rotation=45)
ax.set(ylabel="fraction with data", ylim=(0, 1.05), title="Filter coverage in the clean sample")
ax.grid(axis="x", visible=False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color="C0"),
                   plt.Rectangle((0, 0), 1, 1, color="C1")],
          labels=["JWST/NIRCam", "HST"], loc="lower left")

in_all = np.all([has_data(d, b) for b in use_bands], axis=0)
d = d[in_all].copy()
funnel.append((f"data in all {len(use_bands)} filters", len(d)))
print("filters used:", use_bands)

In [ ]:
print(f"{'step':32s} {'N':>8s}")
for step, n in funnel:
    print(f"{step:32s} {n:8d}")

**Discussion:** we started with the full catalogue and kept a fraction of it. Which step removed the most? Is the final sample still representative of "galaxies in CEERS"? Which kinds of galaxies are we most likely to have lost?

## 4. Features: from fluxes to colours

**Why colours?** A galaxy's redshift is encoded in the *shape* of its spectrum, not its overall brightness. Strong spectral features — the Lyman break at 912/1216 Å and the Balmer/4000 Å break — move through the filters as the redshift increases, and show up as jumps in the colours between neighbouring filters. Colours are ratios of fluxes, so they do not depend on distance or luminosity.

Brightness still carries *some* information (distant galaxies are fainter on average), so we also keep one magnitude.

Let's look at a few galaxies at different redshifts. The dashed lines mark where the 4000 Å break ends up at each galaxy's redshift.

In [ ]:
lam = np.array([wave[b] for b in use_bands])
fig, ax = plt.subplots(figsize=(8, 5))
for k, (zt, col) in enumerate(zip([0.5, 2, 4, 7], ["C0", "C2", "C3", "C1"])):
    i = (d.LP_Z_MED - zt).abs().idxmin()
    zi = d.loc[i, "LP_Z_MED"]
    f = np.array([d.loc[i, f"{b}_FLUX"] for b in use_bands])
    e = np.array([d.loc[i, f"{b}_FLUXERR_EMP"] for b in use_bands])
    norm = d.loc[i, f"{REF_SN}_FLUX"] / 10**k            # offset the SEDs vertically
    ax.errorbar(lam, np.clip(f/norm, 1e-3, None), e/norm, fmt="o-", ms=6, lw=1.5, color=col,
                label=f"z = {zi:.2f}")
    ax.axvline(0.4*(1 + zi), color=col, ls="--", lw=1)
ax.set(xscale="log", yscale="log", xlabel="wavelength [µm]",
       ylabel=r"$f_\nu$ (normalised, offset)", title="Example SEDs from the clean sample")
ax.set_xticks([0.5, 1, 2, 3, 4], ["0.5", "1", "2", "3", "4"])
ax.legend(loc="upper left");

### Non-detections

Many galaxies are not detected in some filters: most importantly, high-redshift galaxies are invisible blueward of the Lyman break — that is *how* we recognise them. A negative or zero flux has no magnitude. Throwing these galaxies away would throw away exactly the high-$z$ objects we care about, so instead we replace any flux below $1\sigma$ with the $1\sigma$ error: an **upper limit**. (Alternatives: asinh magnitudes, flux ratios, or models that handle missing values directly.)

### ✏️ Exercise 3 — Build the feature table

Write `build_features(d, bands, mag_band)` that returns a DataFrame with:

* the colours between **neighbouring** filters, `mag[bands[i]] − mag[bands[i+1]]`, named e.g. `"F115W-F150W"`,
* the magnitude in `mag_band`, named e.g. `"mag_F277W"`,

where the magnitudes are computed from `max(flux, 1σ error)` using the empirical errors.

In [ ]:
def build_features(d, bands, mag_band, floor_sigma=1.0):
    # YOUR CODE HERE
    raise NotImplementedError


feats = build_features(d, use_bands, REF_SN)
z = d["LP_Z_MED"].to_numpy()
print(feats.shape, "  any NaN:", feats.isna().any().any())
feats.describe().round(2)

In [ ]:
# How does each colour change with redshift?
colour_names = [c for c in feats.columns if not c.startswith("mag_")]
ncol = 4
nrow = int(np.ceil(len(colour_names) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2*nrow), sharex=True)
for ax, name in zip(axes.flat, colour_names):
    ax.hexbin(z, feats[name], gridsize=50, bins="log", mincnt=1, cmap="Blues",
              extent=(0, 10, -1.5, 3))
    ax.set_title(name, fontsize=11)
    ax.set_ylim(-1.5, 3)
    ax.grid(False)
for ax in axes.flat[len(colour_names):]:
    ax.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("LP_Z_MED")
fig.suptitle("Each colour vs. redshift", y=1.01)
fig.tight_layout();

In [ ]:
# A colour–colour diagram coloured by redshift
ca, cb = f"{band('150')}-{band('200')}", f"{band('277')}-{band('356')}"
if ca not in feats or cb not in feats:            # fall back to two colours we do have
    ca, cb = colour_names[len(colour_names)//3], colour_names[2*len(colour_names)//3]
order = np.argsort(z)                             # draw high-z on top
fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(feats[cb].to_numpy()[order], feats[ca].to_numpy()[order], c=z[order], s=2,
                cmap=ZCMAP, vmin=0, vmax=8, rasterized=True)
ax.set(xlim=(-1, 2), ylim=(-1, 2.5), xlabel=cb, ylabel=ca, title="Colour–colour diagram")
fig.colorbar(sc, label="LP_Z_MED");

In [ ]:
# Correlations between features
corr = feats.corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=9)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=9)
ax.grid(False)
ax.set_title("Feature correlation matrix")
fig.colorbar(im, label="Pearson r");

**Discussion**

* Which colours change most strongly with redshift, and over which redshift range? Can you connect this to the Balmer and Lyman breaks moving through the filters?
* Is the relation between any single colour and $z$ **linear**? Is it even **single-valued**? What does that suggest about how well a linear model can do?
* Which features are strongly correlated with each other? Recall what we saw about OLS and Lasso with correlated features in the first notebook.

## 5. Linear regression

We split into training and test sets exactly as before, and evaluate with the photo-z metrics from the first notebook:

* **bias** $= \mathrm{median}(\Delta z / (1+z))$
* $\sigma_\mathrm{NMAD} = 1.48 \times \mathrm{median}\left(|\Delta z - \mathrm{median}(\Delta z)| / (1+z)\right)$
* **outlier fraction** $\eta$: fraction with $|\Delta z|/(1+z) > 0.15$

In [ ]:
def photoz_metrics(z_true, z_pred, outlier_cut=0.15):
    z_true, z_pred = np.asarray(z_true), np.asarray(z_pred)
    dz = z_pred - z_true
    return dict(bias=np.median(dz / (1 + z_true)),
                nmad=1.48 * np.median(np.abs(dz - np.median(dz)) / (1 + z_true)),
                outlier_frac=np.mean(np.abs(dz) / (1 + z_true) > outlier_cut),
                rmse=np.sqrt(mean_squared_error(z_true, z_pred)))

def plot_zz(z_true, z_pred, ax, title="", zmax=10):
    ax.hexbin(z_true, z_pred, gridsize=70, bins="log", mincnt=1, cmap="Blues",
              extent=(0, zmax, -1, zmax))
    zz = np.array([0, zmax])
    ax.plot(zz, zz, color="black", lw=1)
    for s in (+1, -1):
        ax.plot(zz, zz + s*0.15*(1 + zz), color="C1", lw=1, ls="--")
    m = photoz_metrics(z_true, z_pred)
    ax.text(0.04, 0.96, f"bias = {m['bias']:+.3f}\n$\\sigma_{{NMAD}}$ = {m['nmad']:.3f}\n"
                        f"η = {m['outlier_frac']:.1%}", transform=ax.transAxes, va="top", fontsize=10)
    ax.set(xlim=(0, zmax), ylim=(-1, zmax), xlabel="LP_Z_MED (label)", ylabel="predicted z", title=title)
    ax.set_aspect("equal")
    ax.grid(False)

results = {}
X = feats.to_numpy()
X_train, X_test, z_train, z_test = train_test_split(X, z, test_size=0.3, random_state=42)
mag_test = X_test[:, -1]                          # the magnitude is the last feature
print(f"train: {len(z_train)}   test: {len(z_test)}")

### ✏️ Exercise 4 — A linear photo-z model

Build a pipeline `lin` with a `StandardScaler` and a `LinearRegression`, fit it on the training set, and predict `zp_lin` for the test set.

In [ ]:
# YOUR CODE HERE


results["linear"] = photoz_metrics(z_test, zp_lin)
fig, ax = plt.subplots(figsize=(6, 6))
plot_zz(z_test, zp_lin, ax, "Linear regression on colours")

In [ ]:
# Which features does the linear model use most? (coefficients on standardised features)
coef = pd.Series(lin[-1].coef_, index=feats.columns)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(coef)), coef.values, color=["C0" if v > 0 else "C1" for v in coef.values])
ax.axhline(0, color="black", lw=1)
ax.set_xticks(range(len(coef)), coef.index, rotation=90, fontsize=9)
ax.set(ylabel="coefficient (Δz per 1σ of feature)", title="Linear model coefficients")
ax.grid(axis="x", visible=False);

**Discussion**

* Look at the shape of the cloud in the prediction plot. At which redshifts is the model biased high, and where biased low? This pattern — predictions squeezed towards the mean of the training set — is called **regression to the mean**, and it is typical of models that are not flexible enough.
* Does the model ever predict **negative** redshifts? Why can a linear model do that?
* Can you interpret the coefficients? Remember that many features are correlated.

## 6. More flexible models

### Polynomial features with Ridge regularization

We give the model more freedom by adding products and powers of the features. With $n$ features, a degree-$d$ polynomial has $\binom{n+d}{d} - 1$ terms: for ~12 features that is ~90 terms at degree 2 and ~450 at degree 3. That is exactly the situation where **regularization** earns its keep. `RidgeCV` picks $\alpha$ by (efficient leave-one-out) cross-validation on the training set.

### ✏️ Exercise 5 — Polynomial Ridge regression

For degree 2 and 3, build the pipeline `StandardScaler → PolynomialFeatures(degree, include_bias=False) → StandardScaler → RidgeCV(alphas=alphas)`, fit it on the training set, and store it in the dictionary `poly_models[degree]`. Print the number of polynomial features and the chosen $\alpha$.

In [ ]:
alphas = np.logspace(-3, 4, 15)
poly_models = {}

# YOUR CODE HERE


fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, deg in zip(axes, [2, 3]):
    zp = poly_models[deg].predict(X_test)
    results[f"poly{deg} + ridge"] = photoz_metrics(z_test, zp)
    plot_zz(z_test, zp, ax, f"Degree-{deg} polynomial + Ridge")

### A better target: $\log(1+z)$

Our metrics measure errors in units of $(1+z)$: an error of 0.3 is excellent at $z = 6$ but terrible at $z = 0.2$. Least squares on $z$, however, weights every error equally, so the fit is dominated by the high-redshift galaxies with large absolute errors — and it happily predicts $z < 0$.

If we fit $y = \log(1+z)$ instead, a constant error in $y$ corresponds to a constant **fractional** error in $(1+z)$, which matches our metrics, and back-transforming with $z = e^y - 1$ can never give $z \le -1$ (negative predictions become rare and small). `TransformedTargetRegressor` does the transformation and back-transformation for us.

### ✏️ Exercise 6 — Fit $\log(1+z)$

Wrap the degree-3 Ridge pipeline in a `TransformedTargetRegressor` with `func=np.log1p` and `inverse_func=np.expm1`. Call it `model_log`, fit it, and predict `zp_log` on the test set.

In [ ]:
# YOUR CODE HERE


results["poly3 + ridge, log(1+z)"] = photoz_metrics(z_test, zp_log)
fig, ax = plt.subplots(figsize=(6, 6))
plot_zz(z_test, zp_log, ax, "Degree-3 + Ridge, fitted in log(1+z)")

In [ ]:
pd.DataFrame(results).T.style.format({"bias": "{:+.4f}", "nmad": "{:.4f}",
                                      "outlier_frac": "{:.1%}", "rmse": "{:.3f}"})

## 7. Where does the model fail?

Summary numbers hide a lot. Let's look at the metrics as a function of **redshift** and of **magnitude**.

In [ ]:
def binned_metrics(x, z_true, z_pred, edges, min_n=30):
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (x >= lo) & (x < hi)
        if s.sum() < min_n:
            rows.append(dict(x=0.5*(lo + hi), n=s.sum(), bias=np.nan, nmad=np.nan, outlier_frac=np.nan))
        else:
            m = photoz_metrics(z_true[s], z_pred[s])
            rows.append(dict(x=0.5*(lo + hi), n=s.sum(), **m))
    return pd.DataFrame(rows)

preds = {"linear": zp_lin, "poly3 + ridge, log(1+z)": zp_log}
colors = {"linear": "C0", "poly3 + ridge, log(1+z)": "C1"}
z_edges = np.arange(0, 10.5, 0.5)
m_edges = np.arange(np.floor(mag_test.min()), np.ceil(mag_test.max()) + 0.5, 0.5)

fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex="col")
for name, zp in preds.items():
    bz = binned_metrics(z_test, z_test, zp, z_edges)
    bm = binned_metrics(mag_test, z_test, zp, m_edges)
    for row, key in enumerate(["bias", "nmad", "outlier_frac"]):
        axes[row, 0].plot(bz.x, bz[key], "o-", ms=5, color=colors[name], label=name)
        axes[row, 1].plot(bm.x, bm[key], "o-", ms=5, color=colors[name], label=name)
for row, lab in enumerate(["bias", r"$\sigma_\mathrm{NMAD}$", "outlier fraction"]):
    axes[row, 0].set_ylabel(lab)
axes[0, 0].axhline(0, color="black", lw=1)
axes[0, 1].axhline(0, color="black", lw=1)
axes[-1, 0].set_xlabel("LP_Z_MED (label)")
axes[-1, 1].set_xlabel(f"{REF_SN} magnitude")
axes[0, 0].legend(fontsize=10)
fig.suptitle("Photo-z performance vs. redshift and magnitude", y=1.0)
fig.tight_layout();

In [ ]:
# How many training galaxies are there at each redshift?
fig, ax = plt.subplots()
ax.hist(z_train, bins=z_edges)
ax.set(yscale="log", xlabel="LP_Z_MED", ylabel="training galaxies", title="The training set is dominated by z < 4");

**Discussion**

* Where are the outliers concentrated? Relate this to the number of training galaxies at each redshift.
* How does the performance depend on magnitude? Why?
* Our cleaning removed faint sources and uncertain labels. If we apply this model to the **whole** catalogue, will the metrics we measured on the test set still hold?

### Bonus — a preview of non-linear models

Polynomial regression is one way to add flexibility. Tree-based methods, which we will meet later in the course, learn the non-linear mapping from colours to redshift directly. As a teaser, here is a gradient-boosted tree model trained on exactly the same features and target:

In [ ]:
gbr = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(max_iter=500, learning_rate=0.1, random_state=0),
    func=np.log1p, inverse_func=np.expm1).fit(X_train, z_train)
zp_gbr = gbr.predict(X_test)
results["gradient boosting, log(1+z)"] = photoz_metrics(z_test, zp_gbr)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plot_zz(z_test, zp_log, axes[0], "Degree-3 + Ridge, log(1+z)")
plot_zz(z_test, zp_gbr, axes[1], "Gradient boosting, log(1+z)")
pd.DataFrame(results).T.style.format({"bias": "{:+.4f}", "nmad": "{:.4f}",
                                      "outlier_frac": "{:.1%}", "rmse": "{:.3f}"})

## 8. How could we do better?

Discuss in groups — which of these would you try first, and why? Which ones change the *model*, and which change the *data*?

**The labels**
* Our target is a LePHARE photo-z, not a spectroscopic redshift. A model trained on these labels can at best reproduce LePHARE, including its mistakes. How could we test our model against spectroscopic redshifts, and what would change if we trained on them instead?
* We dropped galaxies with broad PDFs. Could we use the PDF width as a **weight** instead of a hard cut?

**The features**
* We used only filters with near-complete coverage. How much do the missing HST bands (especially the bluest ones) matter at low redshift?
* We replaced non-detections with $1\sigma$ upper limits. Alternatives: asinh magnitudes, fluxes normalised to one band, or models that accept missing values (the gradient-boosting model above can take `NaN`s directly).
* The flux **uncertainties** tell us how much to trust each colour — can we give them to the model, as features or weights?
* The catalogue also has aperture fluxes (`_FLUX_APER`) in 12 apertures. Would colours from a small aperture be less noisy for faint galaxies? What about size and morphology?

**The target and the model**
* Fitting $\log(1+z)$ helped. Other options: predicting redshift **bins** (classification), or predicting a full **PDF** (quantile regression, mixture density networks).
* Non-linear models: $k$-nearest neighbours, random forests, gradient boosting, neural networks, Gaussian processes.

**The training set**
* The training set is dominated by $z < 4$ and bright galaxies. Could we **reweight** or **stratify** by redshift so rare high-$z$ galaxies are not drowned out?
* Our random train/test split mixes galaxies from the same parts of the sky. Would a **spatial** split (e.g. by pointing) give a more honest estimate of performance on new fields?
* The training sample is not representative of the full catalogue (brighter, cleaner). This **domain shift** is one of the central problems for photo-z's in Euclid and Roman.